# Nguyên

# Dự án: Echo Valley - Dự án Phân tích Thương mại điện tử Olist
**Tên Notebook:** 06_api_web_service.ipynb
**Mục tiêu:** Thử nghiệm logic cho API (FastAPI), tải mô hình đã huấn luyện và giả lập các request dự báo trước khi triển khai chính thức.


# 1. Load mô hình và Preprocessing Pipeline

## 1.1 Tải artifacts

### 1.1.1 Load các file .pkl

**a.** Sử dụng `pickle` hoặc `joblib` để load mô hình và scaler

In [1]:
import os
import joblib

# Thử tìm đúng thư mục chứa models
model_dirs = ['../models', 'models', 'HK3/Echo Valley/models', '../HK3/Echo Valley/models']
model_dir = None
for candidate in model_dirs:
    if os.path.exists(os.path.join(candidate, 'model.pkl')):
        model_dir = candidate
        break
        
if not model_dir:
    raise FileNotFoundError("Không tìm thấy tệp model.pkl! Vui lòng chạy Notebook 05 trước.")

print(f"Đang tải artifacts từ thư mục: {model_dir}")
model = joblib.load(os.path.join(model_dir, 'model.pkl'))
scaler = joblib.load(os.path.join(model_dir, 'scaler.pkl'))
features = joblib.load(os.path.join(model_dir, 'features.pkl'))

print("Mô hình và Scaler đã được tải thành công!")
print(f"Số lượng features mô hình mong đợi: {len(features)}")
print("Danh sách features:", features)


Mô hình và Scaler đã được tải thành công!
Số lượng features mô hình mong đợi: 12
Danh sách features: ['price', 'freight_value', 'distance', 'payment_installments', 'payment_value', 'purchase_hour', 'purchase_day', 'purchase_month', 'purchase_dayofweek', 'payment_type_credit_card', 'payment_type_debit_card', 'payment_type_voucher']


**Nhận xét:**
- [Điền nhận xét: Mô hình đã load thành công chưa? Các tham số có đúng không?]

# 2. Xây dựng logic dự báo (API Logic)

## 2.1 Viết hàm xử lý đầu vào

### 2.1.1 Định nghĩa hàm `predict`

**a.** Hàm nhận JSON input, biến đổi dữ liệu (preprocess) và trả về kết quả dự báo

In [2]:
import pandas as pd
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return 6367 * c

def predict_delivery_time(input_data):
    """
    Hàm nhận input dạng dict chứa thông tin đơn hàng, thực hiện tiền xử lý
    và dự báo thời gian giao hàng (ngày).
    """
    # 1. Tính khoảng cách địa lý
    dist = haversine_distance(
        input_data['customer_lat'], input_data['customer_lng'],
        input_data['seller_lat'], input_data['seller_lng']
    )
    
    # 2. Xử lý thời gian mua hàng
    purchase_time = pd.to_datetime(input_data['order_purchase_timestamp'])
    purchase_hour = purchase_time.hour
    purchase_day = purchase_time.day
    purchase_month = purchase_time.month
    purchase_dayofweek = purchase_time.dayofweek
    
    # 3. Tạo DataFrame khớp với features của Model
    data = {
        'price': [input_data.get('price', 0.0)],
        'freight_value': [input_data.get('freight_value', 0.0)],
        'distance': [dist],
        'payment_installments': [input_data.get('payment_installments', 1.0)],
        'payment_value': [input_data.get('payment_value', 0.0)],
        'purchase_hour': [purchase_hour],
        'purchase_day': [purchase_day],
        'purchase_month': [purchase_month],
        'purchase_dayofweek': [purchase_dayofweek]
    }
    
    # Tạo các cột dummy cho phương thức thanh toán
    payment_type = input_data.get('payment_type', 'credit_card')
    for col in features:
        if col.startswith('payment_type_'):
            type_suffix = col.replace('payment_type_', '')
            data[col] = [1 if payment_type == type_suffix else 0]
            
    df_in = pd.DataFrame(data)
    
    # Đảm bảo đúng thứ tự các cột features
    df_in = df_in[features]
    
    # 4. Scale features
    df_scaled = scaler.transform(df_in)
    
    # 5. Dự báo
    pred = model.predict(df_scaled)[0]
    return float(pred)

print("Đã định nghĩa hàm predict_delivery_time thành công!")

Đã định nghĩa hàm predict_delivery_time thành công!


**Nhận xét:**
- [Điền nhận xét: Hàm dự báo có chạy đúng không? Có gặp lỗi định dạng dữ liệu không?]

# 3. Giả lập Request (Integration Testing)

## 3.1 Kiểm tra phản hồi của mô hình

### 3.1.1 Tạo sample request

**a.** Tạo một input mẫu (ví dụ: tọa độ, loại sản phẩm) để test hàm

In [3]:
# Tạo request giả lập (ở dạng JSON/dict)
sample_request = {
    'customer_lat': -23.5505,
    'customer_lng': -46.6333,
    'seller_lat': -22.9068,
    'seller_lng': -43.1729,
    'order_purchase_timestamp': '2026-07-03 14:30:00',
    'price': 99.90,
    'freight_value': 15.50,
    'payment_installments': 3,
    'payment_value': 115.40,
    'payment_type': 'credit_card'
}

# Gọi hàm dự báo
estimated_days = predict_delivery_time(sample_request)
print("=== KẾT QUẢ GIẢ LẬP REQUEST ===")
print(f"Địa điểm khách hàng (São Paulo): Lat={sample_request['customer_lat']}, Lng={sample_request['customer_lng']}")
print(f"Địa điểm người bán (Rio de Janeiro): Lat={sample_request['seller_lat']}, Lng={sample_request['seller_lng']}")
print(f"Thời gian giao hàng dự báo: {estimated_days:.2f} ngày")

=== KẾT QUẢ GIẢ LẬP REQUEST ===
Địa điểm khách hàng (São Paulo): Lat=-23.5505, Lng=-46.6333
Địa điểm người bán (Rio de Janeiro): Lat=-22.9068, Lng=-43.1729
Thời gian giao hàng dự báo: 9.68 ngày


**Nhận xét:**
- [Điền nhận xét: Kết quả dự báo có logic không?]

# 4. Chuẩn bị cho FastAPI

## 4.1 Sẵn sàng cho triển khai

### 4.1.1 Export logic sang api.py

**a.** Ghi chú các bước cần chuyển sang file `api.py`

In [4]:
# Ghi chú logic cần export sang api.py
api_code = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
import numpy as np

app = FastAPI(title="Olist Delivery Predictor API")

# Load artifacts
model = joblib.load('models/model.pkl')
scaler = joblib.load('models/scaler.pkl')
features = joblib.load('models/features.pkl')

class OrderInput(BaseModel):
    customer_lat: float
    customer_lng: float
    seller_lat: float
    seller_lng: float
    order_purchase_timestamp: str
    price: float
    freight_value: float
    payment_installments: int
    payment_value: float
    payment_type: str

def haversine(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    return 6367 * 2 * np.arcsin(np.sqrt(np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2))

@app.post("/predict")
def predict(order: OrderInput):
    try:
        dist = haversine(order.customer_lat, order.customer_lng, order.seller_lat, order.seller_lng)
        p_time = pd.to_datetime(order.order_purchase_timestamp)
        
        data = {
            'price': [order.price],
            'freight_value': [order.freight_value],
            'distance': [dist],
            'payment_installments': [order.payment_installments],
            'payment_value': [order.payment_value],
            'purchase_hour': [p_time.hour],
            'purchase_day': [p_time.day],
            'purchase_month': [p_time.month],
            'purchase_dayofweek': [p_time.dayofweek]
        }
        
        for col in features:
            if col.startswith('payment_type_'):
                data[col] = [1 if order.payment_type == col.replace('payment_type_', '') else 0]
                
        df_in = pd.DataFrame(data)[features]
        pred = model.predict(scaler.transform(df_in))[0]
        return {"estimated_delivery_time_days": float(pred)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
"""
print("Mã nguồn mẫu của FastAPI đã được lưu trữ trong ghi chú này.")

Mã nguồn mẫu của FastAPI đã được lưu trữ trong ghi chú này.


**Nhận xét:**
- [Điền nhận xét: Mọi logic đã sẵn sàng để code file .py chưa?]